# Building Image Generation Applications (Azure OpenAI)

There's more to LLMs than text generation. You can also generate images from text descriptions. Images as a modality are useful across MedTech, architecture, tourism, game development, marketing, and more. In this lesson we work with today's **GPT Image** models on Microsoft Foundry.

## Learning goals

By the end of this lesson you'll be able to:

- Explain what image generation is and where it's useful.
- Understand the `gpt-image` model family and how it differs from the legacy DALL·E models.
- Build an image generation application and edit images.

## What is image generation?

Image generation models create images from a text prompt. Modern models such as `gpt-image` learn the relationship between text and images during training, then iteratively turn random noise into an image that matches your prompt.

Two well-known families of image models are:

- **`gpt-image` (OpenAI)** — the current generation used in this lesson. It supports text-to-image generation and image editing (inpainting with a mask).
- **Midjourney** — a popular third-party model with its own service and Discord-based workflow.

> The older OpenAI image models — **DALL·E 2** and **DALL·E 3** — are legacy. DALL·E 3 is no longer available for new deployments, and features like `create_variation` only existed in DALL·E 2. Use the `gpt-image` models for new applications.

On Microsoft Foundry, **`gpt-image-2`** is the latest and most capable image model and is the recommended default. `gpt-image-1.5` and `gpt-image-1-mini` are also generally available.

> **Important:** `gpt-image` models return the generated image as **base64** (`b64_json`), not as a URL. Your code decodes the base64 string to bytes and saves it — there's no image URL to download.


## اپنی پہلی امیج جنریشن ایپلیکیشن بنانا

تو ایک امیج جنریشن ایپلیکیشن بنانے کے لیے کیا چاہیے؟ آپ کو درج ذیل لائبریریوں کی ضرورت ہوگی:

- **python-dotenv**, آپ کو سختی سے مشورہ دیا جاتا ہے کہ آپ اپنی رازدار معلومات کو کوڈ سے دور * .env * فائل میں رکھنے کے لیے اس لائبریری کا استعمال کریں۔
- **openai**, یہ لائبریری وہ ہے جو آپ OpenAI API کے ساتھ تعامل کے لیے استعمال کریں گے۔
- **pillow**, پائتھن میں تصاویر کے ساتھ کام کرنے کے لیے۔

اگر پہلے سے نہیں کیا تو، [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) صفحے پر دی گئی ہدایات پر عمل کریں تاکہ Microsoft Foundry ریسورس اور ماڈل بنایا جا سکے۔ ماڈل کے طور پر **gpt-image-2** منتخب کریں (حالیہ Azure OpenAI امیج ماڈل؛ DALL·E پرانا ماڈل ہے)۔

1. ایک فائل *.env* بنائیں جس میں درج ذیل مواد ہو:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

   اپنی ریسورس کے لیے Microsoft Foundry پورٹل میں "Deployments" سیکشن میں یہ معلومات تلاش کریں۔



1. اوپر دی گئی لائبریریاں ایک فائل *requirements.txt* میں اس طرح جمع کریں:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. اگلا، ورچوئل ماحول بنائیں اور لائبریریاں انسٹال کریں:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!نوٹ]
> ونڈوز کے لیے، اپنا ورچوئل ماحول بنانے اور فعال کرنے کے لیے درج ذیل کمانڈز استعمال کریں:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. *app.py* نامی فائل میں درج ذیل کوڈ شامل کریں:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # import dotenv
    dotenv.load_dotenv()

    # Azure OpenAI (Microsoft Foundry) کلائنٹ کو ترتیب دیں۔
    # امیج ماڈلز کو ایک حالیہ API ورژن کی ضرورت ہوتی ہے — اپنے ماڈل کے لیے Foundry دستاویزات چیک کریں۔
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # امیج جنریشن API استعمال کرکے ایک تصویر بنائیں
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # اپنی پرامپٹ ٹیکسٹ یہاں داخل کریں
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # مثلاً "gpt-image-2"
        )
        # محفوظ شدہ تصویر کے لیے ڈائریکٹری سیٹ کریں
        image_dir = os.path.join(os.curdir, 'images')

        # اگر ڈائریکٹری موجود نہیں ہے، تو اسے بنائیں
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # امیج کا راستہ شروع کریں (نوٹ کریں کہ فائل کی قسم png ہونی چاہیے)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image ماڈلز تصویر کو base64 (b64_json) کے طور پر واپس کرتے ہیں، URL کے طور پر نہیں
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # تصویر کو ڈیفالٹ امیج ویور میں دکھائیں
        image = Image.open(image_path)
        image.show()

    # استثنیات کو پکڑیں
    except BadRequestError as err:
        print(err)

    ```

آئیے اس کوڈ کی وضاحت کرتے ہیں:

- سب سے پہلے، ہم وہ لائبریریاں امپورٹ کرتے ہیں جن کی ہمیں ضرورت ہے، جن میں OpenAI لائبریری، dotenv لائبریری، base64 ماڈیول، اور Pillow لائبریری شامل ہیں۔

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- اگلا، ہم *.env* فائل سے ماحول کے متغیرات کو لوڈ کرتے ہیں۔

    ```python
    # ڈاٹ انووٹ کو درآمد کریں
    dotenv.load_dotenv()
    ```

- اس کے بعد، ہم Azure OpenAI (Microsoft Foundry) کلائنٹ کو ترتیب دیتے ہیں۔

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- پھر، ہم تصویر تیار کرتے ہیں:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # یہاں اپنا پرامپٹ متن درج کریں
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    `gpt-image` ماڈلز تصویر کو **base64** سٹرنگ کی شکل میں `data[0].b64_json` میں واپس کرتے ہیں۔ ہم اسے بائٹس میں ڈی کوڈ کرتے ہیں اور فائل میں لکھتے ہیں — آپ کے پاس ڈاؤن لوڈ کرنے کے لیے کوئی URL نہیں ہوتا۔

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- آخر میں، ہم تصویر کو کھولتے ہیں اور اسے دکھانے کے لیے معیاری امیج ویوور استعمال کرتے ہیں:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### تصویر بنانے کی مزید تفصیلات

آئیے `images.generate` کے پیرامیٹرز کو دیکھتے ہیں:

- **prompt**، وہ ٹیکسٹ پرامپٹ ہے جو تصویر بنانے کے لیے استعمال ہوتا ہے۔ یہاں یہ ہے "گھوڑے پر خرگوش، ہنڈی پکڑے ہوئے، دھندلے میدان میں جہاں نیلے زعفران اگ رہے ہیں"۔
- **size**، تیار کردہ تصویر کا سائز ہے (مثلاً `1024x1024`, `1536x1024`, `1024x1536`، یا `"auto"`).
- **n**، وہ تصاویر کی تعداد ہے جو تیار کی گئی ہیں۔ یہاں ہم ایک تصویر بناتے ہیں۔
- **model**، آپ کے تصویر کے ماڈل کی تعیناتی کا نام ہے (مثلاً `gpt-image-2`)۔

> تصویر کے ماڈلز `temperature` پیرامیٹر استعمال نہیں کرتے — یہ ایک ٹیکسٹ جنریشن کنٹرول ہے۔ مختلف تصاویر حاصل کرنے کے لیے API کو دوبارہ کال کریں؛ مختلف تصاویر کی تعداد کم کرنے کے لیے اپنا پرامپٹ زیادہ مخصوص بنائیں۔

## تصویر بنانے کی اضافی خصوصیات

آپ نے دیکھا کہ چند لائنوں پائتھن کے ذریعے تصویر کیسے بنائی جاتی ہے۔ `gpt-image` ماڈلز موجودہ تصویر کو بھی **ترمیم** کر سکتے ہیں۔ ایک تصویر، ایک اختیاری **ماسک** (جو تبدیل ہونے والے حصے کو نشان زد کرتا ہے)، اور ایک پرامپٹ فراہم کر کے آپ تصویر کے حصے کو بدل سکتے ہیں — مثلاً، خرگوش کے سر پر ٹوپی ڈالنا۔

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# ترامیم بھی بیس 64 کے طور پر واپس کی جاتی ہیں
edited_image = base64.b64decode(response.data[0].b64_json)
```

بنیادی تصویر میں صرف خرگوش ہے؛ آخری تصویر میں خرگوش کے سر پر ٹوپی ہے۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
